In [13]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [8]:
df = pd.read_csv("Data\\transformed_data.csv", low_memory=False)

print(df.shape)

(3068095, 18)


In [14]:
from sqlalchemy import create_engine, text

load_dotenv()

db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "")
db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
database_name = os.getenv("DB_NAME", "big_data_assignment")
admin_db = os.getenv("DB_ADMIN_DB", "postgres")

admin_engine = create_engine(
    f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{admin_db}",
    isolation_level="AUTOCOMMIT",
)

with admin_engine.connect() as conn:
    exists = conn.execute(
        text("SELECT 1 FROM pg_database WHERE datname = :name"),
        {"name": database_name}
    ).scalar()
    if not exists:
        conn.execute(text(f"CREATE DATABASE {database_name}"))
        print(f"Created database: {database_name}")

engine = create_engine(
    f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{database_name}"
)

Created database: big_data_assignment


In [15]:
df.to_sql(
    "cybersecurity_logs",
    engine,
    if_exists="replace",
    index=False,
    chunksize=50000
)

print("Data loaded into PostgreSQL successfully!")

Data loaded into PostgreSQL successfully!


In [ ]:
query_df = pd.read_sql("SELECT COUNT(*) FROM cybersecurity_logs", engine)
query_df

,count
0,3068095
